# Train a Twi TTS model from scratch

Trains a Piper VITS voice on IPA phonemes with no Twi checkpoint to start from.

**Read this before choosing this notebook.** From scratch needs **days** of GPU time and
far more data than finetuning. If you are working on Twi, or any language whose phonemes
overlap the published inventory, `finetune.ipynb` will get you a better model in hours.

This notebook is the right choice when:

- you are training a **different language** whose inventory barely overlaps, or
- you want a genuinely independent baseline.

Even then, **warm-start from an English Piper checkpoint** rather than random init. We did:
`libritts_r/medium` gave a working vocoder and speaker machinery for free, and 59 of our
phoneme ids landed on its pretrained rows for the same sounds.


In [ ]:
# A GPU is required. On Colab: Runtime -> Change runtime type -> GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Piper's trainer, plus the two phonemisers. This takes a few minutes.
!apt-get -qq install -y espeak-ng > /dev/null
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git
!pip install -q -e './piper1-gpl[train]'

# ghana-g2p for Twi. Installed from source because africa-g2p's wheel build is broken
# upstream (it ships data/ twice, so hatchling refuses the archive).
!git clone -q https://github.com/AfriSpeech/africa-g2p.git
!git clone -q https://github.com/GhanaNLP/ghana-g2p.git
import sys; sys.path[:0] = ['africa-g2p/src', 'ghana-g2p/src']
from ghana_g2p import GhanaG2P
print('twi g2p:', GhanaG2P('Asante Twi').ipa('Akwaaba', sep=' '))

In [ ]:
# VITS needs a Cython extension that pip does not build, and whose absence only shows up
# as ModuleNotFoundError once training starts. Build it now.
!pip install -q cython
%cd piper1-gpl/src
!python piper/train/vits/monotonic_align/setup.py build_ext --inplace 2>&1 | tail -2
# Upstream's __init__ imports from a nested path the build does not create.
!sed -i 's/^from \\.monotonic_align\\.core import/from .core import/' \
    piper/train/vits/monotonic_align/__init__.py
%cd /content
import torch
sys.path.insert(0, 'piper1-gpl/src')
from piper.train.vits import monotonic_align
print('monotonic_align ok', monotonic_align.maximum_path(
    torch.zeros(1, 3, 5), torch.ones(1, 3, 5)).shape)

In [ ]:
!git clone -q https://github.com/GhanaNLP/stable-twi-tts.git
import sys; sys.path.insert(0, 'stable-twi-tts')
!ls stable-twi-tts/training

## 1. Get the data

[`ghanaopendata/new-twi-tts-aligned-ipa`](https://huggingface.co/datasets/ghanaopendata/new-twi-tts-aligned-ipa) carries audio, text
and IPA together. It is ~23 GB, so on Colab mount Drive or use a subset first.


In [ ]:
from huggingface_hub import snapshot_download

# Start with a few shards to check the pipeline end to end before committing to 23 GB.
SHARDS = 4        # set to None for the whole dataset
repo = 'ghanaopendata/new-twi-tts-aligned-ipa'
patterns = [f'data/train-{i:05d}-*' for i in range(SHARDS)] if SHARDS else None
path = snapshot_download(repo, repo_type='dataset', local_dir='data',
                         allow_patterns=patterns)
!ls data/data | head -5 && du -sh data

## 2. Export wavs and a manifest

One shared wav directory at 22.05 kHz, plus a manifest. 22.05 kHz because the Piper
checkpoints we finetune from are 22.05 kHz — resampling once here beats fighting it later.


In [ ]:
!python stable-twi-tts/training/tts_data.py wavs \
    --data data/data --out tts22k --sr 22050 --threads 8
!head -2 tts22k/manifest.tsv | cut -c1-120

## 3. Phonemise from text

**This is the step that decides whether the model works.** Targets come from the same
phonemiser used at inference — ghana-g2p for Twi, espeak-ng for English — so there is no
train/inference gap by construction.


In [ ]:
!python stable-twi-tts/training/retarget_g2p.py \
    --data tts22k --piper-src piper1-gpl/src
!wc -l tts22k/metadata_train_g2p.csv tts22k/phonemes_g2p.json

## 4. Speaker labels

Skip this if your data has real speaker ids. Ours did not: broadcast corpora are usually
many-speaker and unlabelled, and training a single-speaker model on many voices yields an
averaged, unstable timbre.

Clustering x-vectors manufactures labels. 161k×161k pairwise distances is 100 GB, so this
over-segments with k-means and agglomerates the *centroids* instead. The threshold errs toward
splitting: over-splitting one speaker costs a model almost nothing, merging two muddies a voice.


In [ ]:
!pip install -q speechbrain scikit-learn
!python stable-twi-tts/training/speaker_labels.py \
    --data data/data --embdir spk_emb --out speakers.parquet --threshold 0.7

## 5. Warm-start checkpoint

An English Piper checkpoint gives a trained vocoder and discriminators. Its speaker table is
resized to your speaker count, with new rows seeded from real pretrained voices.


In [ ]:
from huggingface_hub import hf_hub_download
base = hf_hub_download('rhasspy/piper-checkpoints',
                       'en/en_US/libritts_r/medium/epoch=404-step=1887300.ckpt',
                       repo_type='dataset')
NSPK = !cut -d'|' -f2 tts22k/metadata_train_g2p.csv | sort -u | grep -c .
NSPK = int(NSPK[0]); print('speakers:', NSPK)
!python stable-twi-tts/training/adapt_checkpoint.py \
    --ckpt {base} --out warmstart.ckpt --num-speakers {NSPK}

### Graft embeddings for symbols the base model already knows

Where your phoneme exists in the base checkpoint's inventory, copy its learned embedding
instead of starting random. Note the rescale: the two runs have different embedding scales, and
copying raw vectors lands far outside the distribution the encoder expects — ours needed a 3.7×
adjustment.


In [ ]:
# Optional: only if your map adds symbols the base checkpoint also has.
# !python stable-twi-tts/training/graft_embeddings.py \
#     --ckpt warmstart.ckpt --pretrained {base} \
#     --graft tts22k/graft.json --out warmstart_grafted.ckpt

## 6. Train

Full learning rate here, unlike a finetune. Expect **days**, not hours: our bilingual run took
~2 hours per epoch on an H200 at 0.72 it/s over 178k utterances.


In [ ]:
!python -m piper.train fit \
    --data.voice_name twi_scratch \
    --data.csv_path tts22k/metadata_train_g2p.csv \
    --data.audio_dir tts22k/wav \
    --data.dataset_type phoneme_ids \
    --data.phonemes_path tts22k/phonemes_g2p.json \
    --data.espeak_voice en-us \
    --data.phoneme_type text \
    --data.num_symbols 256 \
    --data.cache_dir runs/cache \
    --data.config_path runs/config.json \
    --data.batch_size 32 \
    --data.num_workers 8 \
    --data.validation_split 0.005 \
    --model.num_speakers {NSPK} \
    --model.sample_rate 22050 \
    --model.warmstart_ckpt warmstart.ckpt \
    --model.mos_metric utmos \
    --trainer.default_root_dir runs \
    --trainer.max_epochs -1 \
    --trainer.precision bf16-mixed \
    --trainer.val_check_interval 2000 \
    --trainer.accelerator gpu --trainer.devices 1

## 7. Knowing when to stop

Piper keeps top-5 by `val_mel` (reconstruction, lower better) and top-5 by `val_mos` (UTMOS
perceptual, higher better). **Do not early-stop on `val_mel`** — it saturates while the
adversarial losses are still removing audible artifacts.

Our run, for a sense of shape:

| | `val_mos` | `val_mel` |
|---|---|---|
| step 2k | 2.56 | 0.479 |
| epoch 4 | 2.75 | 0.474 |
| epoch 7 | **3.02** | 0.469 |

`val_mel` got *worse* for two epochs mid-run while the encoder absorbed new symbols, then
recovered. Stopping on it would have fired early.

Treat convergence as *both* `val_mos` and round-trip phoneme error plateauing — then choose
between the surviving checkpoints by listening.


In [ ]:
!python stable-twi-tts/training/synth.py \
    --checkpoint runs/lightning_logs/version_0/checkpoints/last.ckpt \
    --config runs/config.json --manifest tts22k/manifest_val.tsv \
    --out synth --limit 100
!python stable-twi-tts/training/tts_eval.py \
    --manifest tts22k/manifest_val.tsv --synth-dir synth \
    --real-dir tts22k/wav --limit 100

## 8. Export

Same as the finetune notebook — see its final sections for exporting a voice directory and
ranking voices by measured quality rather than by training hours.


In [ ]:
!python stable-twi-tts/tools/export_voice.py \
    --checkpoint runs/lightning_logs/version_0/checkpoints/last.ckpt \
    --train-config runs/config.json --manifest tts22k/manifest.tsv \
    --out voices/twi_scratch --top-n 10 --min-hours 0.5 --lexicon

## Five things that fail silently

Each of these cost this project real time. None of them raise an error — they produce
fluent-sounding wrong audio, which is much more expensive than a crash.

1. **Phonemise with the same function at training and inference.** We trained on
   ASR-derived phonemes and synthesised from G2P-derived ones. They disagreed on 26% of
   units for Twi and 51% for English, and TTS phoneme error tracked it almost exactly
   (25.6% vs 68.6%). Nothing errored.
2. **Clear the phoneme cache when you change the id map.** Piper caches phoneme tensors
   keyed by *text*, not by ids. Change the map and the stale tensors are silently reused,
   so you train on the old targets believing you changed them. Delete `cache/*.phonemes.pt`
   and keep `*.audio.pt` — the audio cache is the expensive one.
3. **Never split IPA by character.** Units like `kʰ`, `t͡ʃ`, `k͡p`, `aɪ` are single symbols.
4. **Don't early-stop on `val_mel`.** It saturates while the adversarial losses are still
   removing artifacts. Keep top-k by `val_mos` too, and measure round-trip phoneme error.
5. **Resize the speaker table before loading a checkpoint with a different speaker count** —
   otherwise the load fails, or worse, silently ignores the speaker id.
